In [42]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import random

# Fijamos semilla para reproducibilidad (muy importante en proyectos serios)
np.random.seed(42)
random.seed(42)

csv_path = Path.cwd() / 'data' / 'raw'/ 'vivino_all_types.csv'

# 1️⃣ Generamos valores numéricos base (0–20)
df = pd.read_csv(csv_path)

n = len(df)

stock_numeric = np.random.randint(0, 21, size=n)

# 2️⃣ Diccionario para convertir algunos números a texto
num_to_text = {
    0: "cero",
    1: "uno",
    2: "dos",
    3: "tres",
    4: "cuatro",
    5: "cinco",
    6: "Seis",       # mayúscula para dirty case
    7: "siete",
    8: "ocho",
    9: "nueve",
    10: "diez"
}

# 3️⃣ Creamos columna como objeto (para permitir mezcla)
df["stock"] = stock_numeric.astype(object)

# 4️⃣ Introducimos dirty values
for i in range(n):
    
    rand = random.random()
    
    # 10% missing
    if rand < 0.10:
        df.loc[i, "stock"] = np.nan
        
    # 15% texto
    elif rand < 0.25:
        value = stock_numeric[i]
        if value in num_to_text:
            df.loc[i, "stock"] = num_to_text[value]
        else:
            df.loc[i, "stock"] = str(value)
        
# Guardamos si querés
df.to_csv("data/raw/vivino_raw.csv", index=False)

In [43]:
csv_path = Path.cwd() / 'data' / 'raw'/ 'vivino_raw.csv'
df = pd.read_csv(csv_path)

display(df.head())
print('shape:', df.shape)
print('\ninfo:')
display(df.info())

na_rate = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
dup_count = df.duplicated().sum()

print('\nMissing %:')
display(na_rate)
print('\nDuplicated rows:', int(dup_count))

,wine_id,wine_name,winery,year,wine_type,rating,num_reviews,price,country,region,style,style_body,style_acidity,intensity,sweetness,tannin,stock
0,77137,Unico (Gran Reserva) 2025,Vega Sicilia,2025,Red,4.7,55779,615.65,España,Ribera del Duero,Ribera del Duero Tinto (España),5.0,3.0,4.0,1.0,4.0,6
1,77136,Unico Reserva Especial Edición 2025,Vega Sicilia,2025,Red,4.7,15151,502.47,España,Ribera del Duero,Ribera del Duero Tinto (España),5.0,3.0,4.0,1.0,4.0,NaN
2,1876294,Pingus 2023,Dominio de Pingus,2023,Red,4.6,7031,1450.00,España,Ribera del Duero,Ribera del Duero Tinto (España),5.0,3.0,4.0,1.0,4.0,14
3,84140,El Nido 2023,Bodegas El Nido,2023,Red,4.7,6556,143.90,España,Jumilla,Tinto (España),4.0,3.0,4.0,1.0,3.0,diez
4,77172,Toro 2015,Pintia,2015,Red,4.5,6288,135.00,España,Toro,Toro Tinto (España),5.0,3.0,4.5,1.0,3.5,7


shape: (7965, 17)

info:
<class 'pandas.DataFrame'>
RangeIndex: 7965 entries, 0 to 7964
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   wine_id        7965 non-null   int64  
 1   wine_name      7965 non-null   str    
 2   winery         7964 non-null   str    
 3   year           7955 non-null   str    
 4   wine_type      7965 non-null   str    
 5   rating         7965 non-null   float64
 6   num_reviews    7965 non-null   int64  
 7   price          7965 non-null   float64
 8   country        7965 non-null   str    
 9   region         7965 non-null   str    
 10  style          7798 non-null   str    
 11  style_body     7798 non-null   float64
 12  style_acidity  7798 non-null   float64
 13  intensity      7798 non-null   float64
 14  sweetness      7012 non-null   float64
 15  tannin         3567 non-null   float64
 16  stock          7184 non-null   str    
dtypes: float64(7), int64(2), str(8)
memory

None


Missing %:


tannin           55.22
sweetness        11.96
stock             9.81
style_acidity     2.10
style             2.10
style_body        2.10
intensity         2.10
year              0.13
winery            0.01
wine_id           0.00
price             0.00
country           0.00
rating            0.00
wine_type         0.00
wine_name         0.00
num_reviews       0.00
region            0.00
dtype: float64


Duplicated rows: 0


In [44]:
def qc_cardinality_and_categories(df: pd.DataFrame, top_n: int = 10, rare_threshold: float = 0.02):
    """
    Prints quick signals for categorical columns:
    - cardinality (nunique)
    - most common categories
    - rare categories (below rare_threshold share)
    """
    n = len(df)
    print(f"Rows: {n:,} | Columns: {df.shape[1]}\n")

    # Candidate categorical columns: object/string/category + low-ish unique ratio
    cat_cols = df.select_dtypes(include=["object", "string", "category", "float"]).columns.tolist()
    if not cat_cols:
        print("No categorical columns detected (object/string/category).")
        return

    summary = []
    for col in cat_cols:
        nunique = df[col].nunique(dropna=True)
        unique_ratio = nunique / max(n, 1)
        summary.append((col, nunique, unique_ratio))

    summary_df = (
        pd.DataFrame(summary, columns=["column", "n_unique", "unique_ratio"])
        .sort_values(["unique_ratio", "n_unique"], ascending=[True, True])
    )

    print("Categorical columns summary (lower unique_ratio usually = more 'categorical'):")
    display(summary_df)

    print("\nDetails:")
    for col, nunique, unique_ratio in summary_df.itertuples(index=False):
        print(f"\n--- {col} ---")
        print(f"n_unique={nunique} | unique_ratio={unique_ratio:.4f}")

        vc = df[col].value_counts(dropna=False)
        print(f"\nTop {top_n} value_counts:")
        display(vc.head(top_n))

        # Rare categories (excluding NaN) by share
        vc_no_na = df[col].value_counts(dropna=True)
        shares = (vc_no_na / vc_no_na.sum())
        rare = shares[shares < rare_threshold]

        if len(rare) > 0:
            print(f"\nRare categories (<{rare_threshold*100:.1f}% of non-missing):")
            display(pd.DataFrame({"count": vc_no_na[rare.index], "share": rare}).sort_values("share"))
        else:
            print(f"\nNo rare categories below {rare_threshold*100:.1f}% threshold.")

# Usage:
qc_cardinality_and_categories(df)

Rows: 7,965 | Columns: 17

Categorical columns summary (lower unique_ratio usually = more 'categorical'):


,column,n_unique,unique_ratio
6,country,1,0.000126
10,style_acidity,3,0.000377
9,style_body,5,0.000628
13,tannin,5,0.000628
3,wine_type,6,0.000753
12,sweetness,7,0.000879
11,intensity,8,0.001004
4,rating,21,0.002637
14,stock,32,0.004018
8,style,39,0.004896



Details:

--- country ---
n_unique=1 | unique_ratio=0.0001

Top 10 value_counts:


country
España    7965
Name: count, dtype: int64


No rare categories below 2.0% threshold.

--- style_acidity ---
n_unique=3 | unique_ratio=0.0004

Top 10 value_counts:


...

,count,share
price,,
32.52,1,0.000126
32.64,1,0.000126
24.35,1,0.000126
95.01,1,0.000126
12.73,1,0.000126
...,...,...
10.50,46,0.005775
9.95,48,0.006026
11.50,49,0.006152



--- wine_name ---
n_unique=7150 | unique_ratio=0.8977

Top 10 value_counts:


wine_name
Verdejo 2024      37
Rosado 2024       31
Blanco 2024       29
Crianza 2020      24
Crianza 2021      20
Albariño 2024     20
Tinto 2021        18
Crianza 2022      17
Godello 2024      17
Cava Brut N.V.    17
Name: count, dtype: int64


Rare categories (<2.0% of non-missing):


,count,share
wine_name,,
Vinyarets Blanco 2022,1,0.000126
La Orquesta 2021,1,0.000126
Viña Calera Verdejo 2023,1,0.000126
Rasa Fonda Chardonnay 2022,1,0.000126
Oriol dels Aspres Blanc 2024,1,0.000126
...,...,...
Crianza 2021,20,0.002511
Crianza 2020,24,0.003013
Blanco 2024,29,0.003641


In [45]:
# ==========================================
#  LIMPIEZA DE "stock"
# ==========================================

# ------------------------------------------
#  Crear diccionario de conversión texto → número
#    Normalizamos en minúscula porque luego
#    estandarizaremos el texto.
# ------------------------------------------

text_to_num = {
    "cero": 0,
    "uno": 1,
    "dos": 2,
    "tres": 3,
    "cuatro": 4,
    "cinco": 5,
    "seis": 6,
    "siete": 7,
    "ocho": 8,
    "nueve": 9,
    "diez": 10
}

df["stock"] = (
    df["stock"]
        .astype(str)                    # Garantiza que todo sea string
        .str.strip()                    # Elimina espacios al inicio y final
        .str.lower()                    # Normaliza mayúsculas/minúsculas
        .replace(text_to_num)           # Convierte palabras a números
        .pipe(pd.to_numeric, errors="coerce")  # Convierte a numérico; errores → NaN
        .fillna(0)                      # Reemplaza missing values por 0
        .astype(int)                    # Convierte definitivamente a integer
)


# ------------------------------------------
# Validaciones finales
# ------------------------------------------

print("\nTipo final:", df["stock"].dtype)
print("Missing values:", df["stock"].isna().sum())
print("Valores mínimos y máximos:")
print("Min:", df["stock"].min())
print("Max:", df["stock"].max())

print("\nDistribución final:")
print(df["stock"].describe())


Tipo final: int64
Missing values: 0
Valores mínimos y máximos:
Min: 0
Max: 20

Distribución final:
count    7965.000000
mean        9.063277
std         6.499103
min         0.000000
25%         3.000000
50%         9.000000
75%        15.000000
max        20.000000
Name: stock, dtype: float64


In [46]:
# ------------------------------------------
# Missings por tipo de vino
# ------------------------------------------

cols = ["sweetness", "style_acidity", "style_body", "intensity", "tannin"]

missing_by_type = (
    df
    .groupby("wine_type")[cols]
    .apply(lambda x: x.isna().mean())
)

missing_by_type

,sweetness,style_acidity,style_body,intensity,tannin
wine_type,,,,,
Dessert,1.000000,1.000000,1.000000,1.000000,1.0
Fortified,0.125000,0.125000,0.125000,0.125000,1.0
Red,0.000000,0.000000,0.000000,0.000000,0.0
Rose,0.000000,0.000000,0.000000,0.000000,1.0
Sparkling,1.000000,0.000000,0.000000,0.000000,1.0
White,0.000414,0.000414,0.000414,0.000414,1.0


Hay missings estructurales (tiene sentido que falten valores), como por ejemplo los taninos que solamente están presentes en los vinos tintos. Los faltantes para vinos fortificados y de postre los vamos a ignorar ya que son espceciales (también son menos).

In [47]:
white_df = df[df["wine_type"] == "White"]

n_white = len(white_df)

missing_white = white_df["sweetness"].isna().sum()

print("Total White:", n_white)
print("Missing en White:", missing_white)
print("Proporción:", missing_white / n_white)

Total White: 2414
Missing en White: 1
Proporción: 0.00041425020712510354


Observamos que hay un solo vino que no fue completado.

In [48]:
white_df[white_df["sweetness"].isna()].T

,2538
wine_id,9250697
wine_name,Unanimous 'Santa Cruz' Albillo Mayor 2022
winery,Tres Piedras
year,2022
wine_type,White
rating,4.3
num_reviews,203
price,28.95
country,España
region,Ribera del Duero


Les asignamos a las variables numéricas faltantes de este vino el promedio de los vinos blancos de Ribera del Duero. Para el "style" le asignamos el valor que más veces aparezca en ese grupo de vinos.

In [49]:
# ------------------------------------------
# Rellenar el vino con missing
# ------------------------------------------

wine_missing = df[df["wine_id"] == 9250697]

# Confirmamos columnas a imputar
num_cols = ["style_body", "style_acidity", "intensity", "sweetness"]
cat_cols = ["style"]

wine_missing[num_cols + cat_cols]

# Filtramos el grupo
group = df[
    (df["wine_type"] == "White") &
    (df["region"] == "Ribera del Duero")
]

# Media para columnas numéricas
group_means = group[num_cols].mean()

# Moda para la columna categórica
group_mode = group["style"].mode()
if len(group_mode) > 0:
    group_mode = group_mode[0]  # Tomamos el valor más frecuente
else:
    group_mode = np.nan  # Por si el grupo está vacío

# Ubicamos el índice del vino faltante
idx = df[df["wine_id"] == 9250697].index[0]

# Imputamos numéricas
for col in num_cols:
    df.loc[idx, col] = group_means[col]

# Imputamos categórica
df.loc[idx, "style"] = group_mode

# Verificamos que se haya completado correctamente

df.loc[idx, ["style_body", "style_acidity", "intensity", "sweetness", "style"]]

style_body            3.030303
style_acidity         2.060606
intensity             2.969697
sweetness             1.030303
style            Spanish White
Name: 2538, dtype: object

Corroboramos si el año que aparece en “wine_name” coincide siempre con “year” → en caso afirmativo, lo eliminamos para que no haya redundancia.

In [50]:
import re

#Tenemos que convertir la variable "year" a integer porque viene como string.

df["year"] = (
    pd.to_numeric(df["year"], errors="coerce")
      .astype("Int64")
)

# Extraer año de 4 dígitos
df["year_from_name"] = df["wine_name"].str.extract(r'(19\d{2}|20\d{2})')

# Convertir a numérico
df["year_from_name"] = (
    pd.to_numeric(df["year_from_name"], errors="coerce")
      .astype("Int64")
)

print("Total filas:", len(df))

print("\nFilas sin año en wine_name:")
print(df["year_from_name"].isna().sum())

print("\nFilas donde no coincide:")
print((df["year_from_name"] != df["year"]).sum())

Total filas: 7965

Filas sin año en wine_name:
638

Filas donde no coincide:
12


In [51]:
mismatch_df = df[
    (df["year_from_name"].notna()) &
    (df["year_from_name"] != df["year"])
]

mismatch_df[[
    "wine_id",
    "wine_name",
    "year",
    "year_from_name",
    "winery",
    "country",
    "region",
    "rating",
    "num_reviews"
]].sort_values("year")

,wine_id,wine_name,year,year_from_name,winery,country,region,rating,num_reviews
5644,2609014,Añada Palo Cortado 1987 1991,1991,1987,Gonzalez-Byass,España,Jerez-Xérès-Sherry,4.5,51
97,1815148,1902 Centenary Carignan Priorat 2017,2017,1902,Mas Doix,España,Priorato,4.6,86
129,6900561,1903 Centenary Grenache 2017,2017,1903,Mas Doix,España,Priorato,4.5,29
106,9724334,1902 Centenary Carignan Tossal D'En Bou Gran V...,2019,1902,Mas Doix,España,Priorato,4.6,58
113,10813230,1903 Garnatxa Velles Vinyes Coma de Cases 2019,2019,1903,Mas Doix,España,Priorato,4.5,46
4294,6332557,Cava Mitic 1908 Gran Reserva Brut Nature 2019,2019,1908,Oriol Rossell,España,Cava,4.0,52
6929,10676400,MarLa Vi de Paratge - Carinyenes des de 1902 2021,2021,1902,Sandra Doix Celler,España,Priorato,4.4,127
4638,2559473,Cava 1907 Colomer Costa Reserva Brut 2022,2022,1907,Colomer,España,Cava,3.9,174
6624,8862548,Pena Fión Viña de 1930 2022,2022,1930,Abadia da Cova,España,Ribeira Sacra,4.3,65
6240,12300365,MarLa Vi de Paratge Garnatxes Des de 1955 2022,2022,1955,Sandra Doix Celler,España,Priorato,4.2,31


In [52]:
# Extraer TODOS los años de 4 dígitos
df["year_candidates"] = df["wine_name"].str.findall(r'(?:19|20)\d{2}')

# Tomar el último año encontrado
df["year_from_name"] = (
    df["year_candidates"]
        .apply(lambda x: x[-1] if len(x) > 0 else None)
)

# Convertir a numérico
df["year_from_name"] = pd.to_numeric(df["year_from_name"], errors="coerce")

mismatch_df = df[
    (df["year_from_name"].notna()) &
    (df["year_from_name"] != df["year"])
]

print("Filas donde no coincide:", len(mismatch_df))

Filas donde no coincide: 0


In [53]:
# Eliminar el año de 4 dígitos al final del wine_name

# Definimos patrón:
# - \s? → espacio opcional antes del año
# - (19\d{2}|20\d{2}) → año de 1900-2099
# - $ → solo si está al final
pattern = r'\s?(19\d{2}|20\d{2})$'

# Aplicamos reemplazo
df["wine_name"] = df["wine_name"].str.replace(pattern, '', regex=True).str.strip()
# Mostramos algunos casos para confirmar
df[["wine_name", "year"]].head(10)

,wine_name,year
0,Unico (Gran Reserva),2025
1,Unico Reserva Especial Edición,2025
2,Pingus,2023
3,El Nido,2023
4,Toro,2015
5,La Nieta,2023
6,Rioja Gran Reserva 904,2015
7,Ribera del Duero,2017
8,Valbuena 5º (Reserva),2016
9,Bruto,2023


In [ ]:
# Eliminar el (España) de la columna "style" y la columna "country"

# Reemplazo usando str.replace y strip
df["style"] = df["style"].str.replace(r'\(España\)', '', regex=True).str.strip()
df.drop(columns=["country","year_from_name","year_candidates"], inplace=True)

In [63]:
# Seleccionamos solo filas donde 'year' es NaN
missing_year_df = df[df["year"].isna()]

# Mostramos las filas completas
missing_year_df

,wine_id,wine_name,winery,year,wine_type,rating,num_reviews,price,region,style,style_body,style_acidity,intensity,sweetness,tannin,stock
199,6266660,Mucho Más Tinto N.V.,Félix Solís,<NA>,Red,4.1,111130,5.55,Vino de España,Tinto,4.0,3.0,4.0,1.0,3.0,10
204,5445838,The Guv’nor N.V.,Félix Solís,<NA>,Red,4.1,26368,5.50,Valdepeñas,Tempranillo,4.0,2.0,4.5,1.0,4.0,5
223,9240299,Mucho Más Tinto - Black Edition N.V.,Félix Solís,<NA>,Red,4.1,7410,6.75,Vino de España,Tinto,4.0,3.0,4.0,1.0,3.0,17
236,12200870,Mucho Mas Gold N.V.,Félix Solís,<NA>,Red,4.0,5874,9.99,Vino de España,Tempranillo,4.0,2.0,4.5,1.0,4.0,4
258,9888252,El Cortez Xo Extra Ordinario N.V.,Torre Oria,<NA>,Red,4.2,4567,9.95,Valencia,Monastrell,5.0,3.0,3.5,1.0,4.5,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6099,1144010,Dry Sack Fino N.V.,Williams & Humbert,<NA>,Fortified,3.0,162,8.90,Jerez-Xérès-Sherry,Spanish Fino Sherry Fortified,2.0,2.0,1.0,1.0,NaN,13
6100,1412680,San Patricio Jerez Seco N.V.,Garvey,<NA>,Fortified,3.1,92,5.45,Jerez-Xérès-Sherry,Sherry,4.0,3.0,4.0,4.0,NaN,0
6101,1453899,Superior N.V.,Antonio Bandeira,<NA>,Fortified,3.1,49,9.50,Penedès,NaN,NaN,NaN,NaN,NaN,NaN,0
6102,6645195,Don José Oloroso N.V.,Romate,<NA>,Fortified,3.4,45,16.13,Jerez-Xérès-Sherry,Spanish Oloroso Sherry Fortified,4.0,2.0,4.0,1.0,NaN,6


No parecería que haya un patrón en los missings, sino más bien algo estructural (vinos en los que no se indica la añada).